# Chapter 7: Reconnaissance and OSINT

> "Give me six hours to chop down a tree and I will spend the first four sharpening the axe."
> Abraham Lincoln (adapted as a recon maxim)

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Distinguish passive from active reconnaissance and explain when each is appropriate.
2. Use public sources to enumerate an organisation's external attack surface.
3. Gather DNS records, WHOIS data, and certificate transparency logs for a target domain.
4. Use Shodan and similar search engines to identify exposed services.
5. Perform OSINT on a target organisation using LinkedIn, job postings, and social media.
6. Enumerate subdomains using multiple techniques.
7. Assess the external attack surface and identify high-value recon findings.

## Key Terms

- **Passive recon**: gathering intelligence without touching target systems.
- **Active recon**: interacting with target systems; leaves traces in logs.
- **OSINT**: Open-Source Intelligence; intelligence from publicly available sources.
- **WHOIS**: protocol/database for registrant information about domains and IP blocks.
- **ASN**: Autonomous System Number; identifies an organisation's IP routing block.
- **Certificate Transparency (CT)**: public logs of all TLS certificates issued.
- **Shodan**: search engine that indexes internet-connected devices and services.
- **Subdomain enumeration**: discovering subdomains of a target domain.
- **Google dorking**: using advanced search operators to find sensitive information.
- **Wayback Machine**: Internet Archive service that caches historical web pages.
- **theHarvester**: tool for gathering emails, subdomains, and hosts from public sources.

---

## The Intelligence Value of Reconnaissance

Reconnaissance is the phase that determines the quality of everything that follows. An attacker or
tester who spends two hours on reconnaissance before touching a target will outperform one who scans
immediately, because they understand the organisation's technology stack, naming conventions,
personnel, and third-party relationships before triggering any alerts.

### Why Passive Recon Comes First

Passive recon leaves no traces in the target's logs because it uses public sources rather than
interacting with target systems directly. It reveals: external IP ranges, domain names and
subdomains, email naming conventions, employee names and roles, technology choices (inferred from
job postings), third-party dependencies (inferred from DNS and certificate records), and sometimes
exposed credentials (from breach databases).

#### The Cost of Skipping Recon

Scanners generate noise. An attacker who scans without first understanding the target may trigger
alerts, exhaust IP addresses, or miss the actual entry point (a forgotten subdomain, a staging
environment, a vendor's VPN gateway). Intelligence gathered in passive recon focuses subsequent
active techniques precisely.

---

## Passive Reconnaissance Techniques

### DNS Enumeration

DNS is a gold mine of passive intelligence. The standard record types and what they reveal:

- **A / AAAA**: maps hostnames to IPv4/IPv6 addresses; reveals every publicly named host.
- **MX**: mail exchangers; identifies the mail provider (Google Workspace, Microsoft 365,
  self-hosted) and often reveals the organisation's internal mail domain name.
- **TXT**: SPF, DKIM, and DMARC records; DMARC policy reveals email security maturity.
  Also used for domain verification strings that reveal cloud services in use (Google Search
  Console, Atlassian, Salesforce, etc.).
- **NS**: authoritative name servers; identifies the DNS provider or registrar.
- **CNAME**: aliases; reveal the actual hosting provider behind a friendly name
  (target.s3.amazonaws.com, target.azurewebsites.net).

#### Zone Transfer (AXFR)

A DNS zone transfer copies all records from a primary to a secondary name server. Misconfigured
name servers that allow zone transfers from any source hand the entire DNS zone to any requester.
`dig AXFR @ns1.target.com target.com` tests for this. Zone transfer misconfiguration is rare on
modern public DNS but common on internal name servers.

### WHOIS and ASN Lookup

WHOIS databases contain registration information for domains and IP blocks. For newer domains under
GDPR, the registrant contact is often redacted, but the registrar, registration date, name servers,
and expiry date remain. IP WHOIS (via ARIN, RIPE, APNIC) reveals the ASN, the organisation name,
the physical address of the address block, and the abuse contact.

#### Pivoting via ASN

Once an ASN is identified, it reveals all IP prefixes announced by that organisation. An attacker
who knows the ASN can enumerate the entire public IP footprint without querying the target directly.
Tools like `bgp.he.net` and `ipinfo.io` provide this lookup.

### Certificate Transparency Logs

Every publicly trusted TLS certificate is logged to Certificate Transparency (CT) logs by the
issuing CA. CT logs are public and searchable. `crt.sh` queries all major CT logs by domain name
and returns every certificate ever issued, including:
- Subdomains listed as Subject Alternative Names (SANs).
- Historical subdomains for decommissioned systems that may still be accessible.
- Third-party services (e.g., mail.target.com issued by SendGrid reveals the email provider).

CT log enumeration is entirely passive and reveals subdomains that DNS enumeration misses because
they are not publicly resolvable but do have valid certificates.

### Shodan and Attack-Surface Search Engines

Shodan continuously scans the internet and indexes banners, service versions, and certificate data
for every reachable host. A Shodan query for an organisation's IP ranges reveals:
- Exposed RDP (3389), SSH (22), Telnet (23), FTP (21) services.
- Industrial control systems and IoT devices.
- Misconfigurations (default credentials in banners, exposed admin panels).
- Software versions that can be cross-referenced with CVE databases.

#### Other Search Engines

Censys and FOFA offer similar capabilities with different indexing. Shodan facets allow filtering
by ASN, country, port, and product. `shodan domain target.com` reveals all hosts Shodan has indexed
under a domain, often revealing subdomains not found via DNS.

### Google Dorking

Advanced Google search operators surface sensitive files and misconfigured pages:

| Dork | Finds |
|---|---|
| `site:target.com filetype:pdf` | PDF documents hosted on the target |
| `site:target.com inurl:admin` | Admin panels |
| `site:target.com "index of"` | Open directory listings |
| `site:target.com ext:env OR ext:sql OR ext:log` | Exposed config or log files |
| `"@target.com" filetype:xlsx` | Spreadsheets mentioning target email addresses |

---

## OSINT on Personnel

### LinkedIn and Social Media

LinkedIn reveals: employee names, titles, departments, tenure, and sometimes direct-report
relationships. Technology choices inferred from job descriptions (if they hire for Kubernetes and
Terraform, their infrastructure is cloud-native). Alumni networks reveal former employees who may
have residual access or institutional knowledge that can be elicited.

#### Email Naming Convention

Five or six email addresses from a breach database or CT log establish the naming convention
(`first.last@`, `flast@`, `f.last@`). This enables construction of a valid email list for the
entire organisation, which is the precursor to spear phishing and password-spray attacks.

### Job Postings

Job postings are inadvertent technical disclosure. "3+ years of experience with Palo Alto Networks
and Fortinet" reveals the network equipment. "Proficiency in Azure Active Directory and Okta" reveals
the identity stack. "Experience with Nessus and Splunk" reveals the security tooling. An attacker
reads job postings to understand the target's technology landscape without touching any system.

---

## Subdomain Enumeration

### Dictionary-Based Brute Force

Tools like `gobuster dns`, `ffuf`, and `amass` send DNS queries for combinations of a wordlist
with the target domain. A good wordlist (SecLists, JHaddix's `all.txt`) covers common subdomains
(api, dev, staging, admin, vpn, mail, internal, test). Slow, spread-out queries blend with normal
recursive DNS traffic.

### Passive Sources

`amass enum -passive -d target.com` queries CT logs, DNS databases, and threat intelligence feeds
without sending a single packet to the target. `theHarvester -d target.com -b all` aggregates
results from search engines, LinkedIn, PGP key servers, and threat feeds. Passive subdomain
enumeration routinely finds dozens of hosts that active scanning would miss.

---

## Why This Matters

Recon is the phase where attackers find entry points that defenders have forgotten exist. A staging
environment with default credentials, a developer's exposed S3 bucket, an old VPN gateway running
unpatched software, a subdomain pointing to an unclaimed cloud service all of these are found
during recon, not during scanning. Defenders who run their own recon programme, continuously
monitoring their external attack surface, catch these exposures before attackers do.

---

## News in Focus

A recurring pattern in breach post-mortems is that initial access was through a forgotten or
shadow-IT asset, a VPN instance not in the asset inventory, a marketing microsite on a subdomain
maintained by an agency, or a test environment inadvertently exposed to the internet. These are
passive-recon findings; the attacker used CT logs or Shodan to find them, never triggering a
network-based detection. External attack surface management (EASM) tools automate continuous
monitoring of these exposures.

---


In [1]:
# Chapter 7 -- Worked Example: passive recon simulation
import hashlib, json

def dns_mock(domain):
    # Simulates DNS record lookup results for a fictional target.
    records = {
        "A":    ["203.0.113.10", "203.0.113.11"],
        "MX":   ["10 mail.example-corp.com", "20 mail2.example-corp.com"],
        "TXT":  ["v=spf1 include:_spf.google.com ~all",
                 "google-site-verification=abc123",
                 "MS=ms12345678",
                 "v=DMARC1; p=quarantine; rua=mailto:dmarc@example-corp.com"],
        "NS":   ["ns1.cloudflare.com", "ns2.cloudflare.com"],
        "CNAME":["www -> example-corp.github.io",
                 "mail -> aspmx.l.google.com"],
    }
    return records

def analyse_txt_records(txts):
    inferences = []
    for txt in txts:
        if "google.com" in txt and "spf1" in txt:
            inferences.append("Email hosted: Google Workspace (SPF includes Google)")
        if "google-site-verification" in txt:
            inferences.append("Google Search Console verified (web presence)")
        if "MS=" in txt:
            inferences.append("Microsoft 365 domain verification present")
        if "DMARC1" in txt:
            policy = txt.split("p=")[1].split(";")[0] if "p=" in txt else "none"
            inferences.append(f"DMARC policy: {policy}")
    return inferences

domain = "example-corp.com"
records = dns_mock(domain)

print(f"=== DNS Recon: {domain} ===\n")
for rtype, values in records.items():
    print(f"  {rtype:6}: {values}")

print("\n=== Intelligence Inferred from TXT Records ===")
for inf in analyse_txt_records(records["TXT"]):
    print(f"  [+] {inf}")

print("\n=== Simulated CT Log Subdomains (crt.sh) ===")
ct_subdomains = [
    "www.example-corp.com",
    "mail.example-corp.com",
    "vpn.example-corp.com",
    "staging.example-corp.com",
    "api.example-corp.com",
    "legacy-crm.example-corp.com",
    "hr.example-corp.com",
    "admin.example-corp.com",
]
high_value = ["vpn","staging","legacy","admin","hr","api"]
for sub in ct_subdomains:
    label = sub.split(".")[0]
    flag = " <-- HIGH VALUE" if any(h in label for h in high_value) else ""
    print(f"  {sub}{flag}")

print("\n=== Attack Surface Summary ===")
print(f"  Unique subdomains found : {len(ct_subdomains)}")
print(f"  Email provider         : Google Workspace")
print(f"  DNS provider           : Cloudflare")
print(f"  Static hosting hint    : GitHub Pages (CNAME)")
print(f"  Microsoft 365 tenant   : Yes (verify domain record)")
print(f"  DMARC policy           : quarantine (phishing harder but not impossible)")


=== DNS Recon: example-corp.com ===

  A     : ['203.0.113.10', '203.0.113.11']
  MX    : ['10 mail.example-corp.com', '20 mail2.example-corp.com']
  TXT   : ['v=spf1 include:_spf.google.com ~all', 'google-site-verification=abc123', 'MS=ms12345678', 'v=DMARC1; p=quarantine; rua=mailto:dmarc@example-corp.com']
  NS    : ['ns1.cloudflare.com', 'ns2.cloudflare.com']
  CNAME : ['www -> example-corp.github.io', 'mail -> aspmx.l.google.com']

=== Intelligence Inferred from TXT Records ===
  [+] Email hosted: Google Workspace (SPF includes Google)
  [+] Google Search Console verified (web presence)
  [+] Microsoft 365 domain verification present
  [+] DMARC policy: quarantine

=== Simulated CT Log Subdomains (crt.sh) ===
  www.example-corp.com
  mail.example-corp.com
  vpn.example-corp.com <-- HIGH VALUE
  staging.example-corp.com <-- HIGH VALUE
  api.example-corp.com <-- HIGH VALUE
  legacy-crm.example-corp.com <-- HIGH VALUE
  hr.example-corp.com <-- HIGH VALUE
  admin.example-corp.com <-- 

## Review Questions (MCQ)

**Q1.** Passive reconnaissance differs from active reconnaissance in that passive recon:
A. Is faster  B. Does not interact with target systems and leaves no logs on them  C. Uses only commercial tools  D. Only covers publicly listed domains

**Q2.** Certificate Transparency logs are useful for:
A. Breaking TLS encryption  B. Discovering subdomains listed as SANs in historical certificates  C. Brute-forcing passwords  D. Identifying open ports

**Q3.** An MX record pointing to `aspmx.l.google.com` reveals:
A. The target uses self-hosted email  B. The target's email is hosted on Google Workspace  C. The target's web server is Google-hosted  D. The target uses DMARC

**Q4.** A job posting requiring "experience with Palo Alto Networks and Okta" reveals:
A. Nothing useful for an attacker  B. The network firewall and identity provider in use  C. The salary budget  D. The number of employees

**Q5.** The `site:target.com filetype:pdf` dork finds:
A. Open directory listings  B. PDF documents hosted on the target  C. Exposed admin panels  D. Email addresses

**Q6.** Shodan is best described as:
A. A vulnerability scanner  B. A search engine indexing internet-connected devices and their banners  C. A password cracker  D. A phishing framework

**Q7.** Which DNS record type reveals the organisation's email security policy (p=reject/quarantine/none)?
A. MX  B. SPF TXT  C. DMARC TXT  D. NS

**Q8.** A DNS zone transfer (AXFR) that succeeds means:
A. The target has strong DNS security  B. The name server reveals its entire zone to any requester  C. DNSSEC is enabled  D. The domain has expired

**Q9.** Email naming convention is most easily determined from:
A. Scanning the mail server  B. Breach database samples or CT log certificate subjects  C. Social engineering the receptionist  D. Running Nmap

**Q10.** External Attack Surface Management (EASM) tools are used by defenders to:
A. Block Shodan's scanner  B. Continuously monitor their own public exposure as an attacker would  C. Replace penetration testing entirely  D. Encrypt DNS traffic

*Answers: Q1 B, Q2 B, Q3 B, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 B, Q10 B.*

## Lab Assignment

**Part A -- DNS recon**: For a domain you own or are authorised to test, run `dig A`, `dig MX`,
`dig TXT`, `dig NS`, and `dig AXFR` (zone transfer attempt). For each result, document what
intelligence it provides about the organisation's technology stack.

**Part B -- CT log search**: Search `crt.sh` for a public organisation's domain (use a large company
with many subdomains for educational value). List 10 subdomains found. For three of them, explain
what the subdomain name implies about the service and why it might be a high-value recon finding.

**Part C -- OSINT dossier**: Build a passive OSINT dossier on a public organisation (a large company
you are not targeting). Include: IP ranges (from ASN lookup), email provider (from MX), identity
provider (from TXT), cloud provider hints (from CNAME), and three inferences from job postings about
their technology stack. Do not interact with any of their systems.

**Part D -- Threat model**: Based on the dossier from Part C, identify the three highest-value
attack paths an external attacker would prioritise and explain why, referencing the recon data.

## References

```{bibliography}
:filter: docname in docnames
```
